# How the 2D segmentation works

Where a staining also fills the nuclear envelope, a section through a nucleus shows a bright ring around a dark
inside. In the superlevel filtration of the image (the cubical persistence forest of the negated image) every
nucleus is a **loop**. It is born when the threshold drops below the weakest point of the ring and dies when it drops
below the brightest point inside. Its **lifetime** (ring minus inside, in units of the 99.8th intensity percentile)
measures how clean the ring is.

Steps (code in `src/nucseg/segment2d.py`):

1. **Projections.** The mean of `window` consecutive slices, for windows stepping through `z_first` .. `z_last`.
   Different nuclei are sharpest at different depths.
2. **Forests.** Each projection is binned (`bin_factor`), smoothed (`smooth_um`), scaled and negated, and a cubical
   persistence forest is built.
3. **Candidates.** Bars with lifetime ≥ `min_lifetime`. Each region is the inside of the bar's cycle representative
   just after birth (`cycle_position`), with holes filled. Candidates are dropped if their area is outside
   `min_area_um2`-`max_area_um2`, if they are narrower than `min_width_um` (twice the radius of the largest disk
   inside), or if their solidity (area / convex-hull area) is below `min_solidity`. The solidity filter removes
   irregular dark pockets in the bright perinuclear ER.
4. **Selection across projections.** Candidates are taken in decreasing lifetime × area and kept unless they overlap
   a kept one. Each nucleus comes from the projection with a clean ring around its largest section; lifetime alone
   would prefer small sections through the top of a nucleus.
5. **Merging** (`merge=True`). An envelope fold reaching across a nucleus can cut its section into pieces. Pieces
   separated by a thin wall (≤ `wall_um`) are merged, unless the merged outline pinches in at both ends of the wall
   (a waist): two nuclei lying against each other.
6. **Outline cleaning** (`clean_outlines=False`). Dark gaps in the ER next to the envelope attach spurs to some
   outlines. With cleaning, the loops are found again on projections smoothed with `clean_smooth_um`, where those
   gaps are closed, and that smooth outline is grown back towards the original one by at most `clean_reach_um`.
7. **Nuclei at the image edge** (`border_frame=False`). A nucleus cut by the image edge forms no loop, unless a
   bright one-pixel frame around the image closes its ring.

All lengths and areas are in µm, so the parameters do not depend on the pixel size or `bin_factor`.

This notebook looks at one stack: the first TIFF in `data/`, or the synthetic example.

In [ ]:
from dataclasses import replace
from pathlib import Path
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from persforest import PersistenceForest

import nucseg
from nucseg import plots
from nucseg.segment2d import prepare

In [ ]:
DATA_DIR = Path("data")
CHANNEL = None
UM_PER_PIXEL = None

paths = nucseg.find_stacks([DATA_DIR]) if DATA_DIR.exists() else []
path = paths[0] if paths else nucseg.write_synthetic_stack(Path("example_data") / "synthetic.tif")
stack = nucseg.read_stack(path, channel=CHANNEL, um_per_pixel=UM_PER_PIXEL)
z_step = "unknown" if stack.z_step_um is None else f"{stack.z_step_um:.3f} µm"
print(f"{stack.name}: {stack.data.shape[0]} slices of {stack.data.shape[1]} x {stack.data.shape[2]} px, "
      f"{stack.um_per_pixel:.4f} µm/px, z step {z_step}")

In [ ]:
parameters = nucseg.Parameters(z_first=None, z_last=None, bin_factor=4)
parameters

## Lifetimes: which loops are nuclei

The barcode of one projection: every bar is a loop of the bright structures. Nuclei have long bars, the meshwork of
the ER many short ones. `min_lifetime` (dashed) separates them; the area, width and solidity filters remove the long
bars that are not nuclei.

In [ ]:
window = parameters.windows(len(stack.data))[len(parameters.windows(len(stack.data))) // 2]
image = prepare(stack.data[window[0]:window[1]].mean(axis=0), stack.um_per_pixel, parameters)
forest = PersistenceForest.from_cubical_grid(-image)
lifetimes = np.sort([bar.death - bar.birth for bar in forest.barcode])[::-1]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].imshow(image, cmap="gray", vmin=0, vmax=1)
axes[0].set_title(f"prepared projection of slices {window[0]}-{window[1] - 1}")
axes[0].axis("off")
axes[1].semilogy(np.arange(1, len(lifetimes) + 1), lifetimes, ".", markersize=3)
axes[1].axhline(parameters.min_lifetime, color="red", linestyle="--", label="min_lifetime")
axes[1].set(xscale="log", xlabel="bar rank", ylabel="lifetime", title=f"{len(lifetimes)} bars")
axes[1].legend()
fig.tight_layout()
plt.show()

## Segmentation

In [ ]:
segmentation = nucseg.segment_2d(stack, parameters)
print(f"{len(segmentation.candidates)} candidates, {len(segmentation.pieces)} selected loops, "
      f"{len(segmentation.nuclei)} nuclei after merging")

## Which projection each nucleus comes from

All candidates per projection (red) and the selected loops taken from that projection (lime). Several nuclei appear
in more than one projection; the one with the largest lifetime × area wins.

In [ ]:
windows = list(segmentation.images)
fig, axes = plt.subplots(1, len(windows), figsize=(4.6 * len(windows), 5), squeeze=False)
for ax, window in zip(axes[0], windows):
    image = segmentation.images[window]
    title = f"z {window[0]}-{window[1] - 1}"
    plots.plot_regions(ax, segmentation, [r for r in segmentation.candidates if r.window == window], image=image,
                       annotate=False, color="red", linewidth=0.8, title=title)
    plots.plot_regions(ax, segmentation, [r for r in segmentation.pieces if r.window == window], image=image,
                       annotate=False, color="lime", linewidth=1.5, title=title)
fig.tight_layout()
plt.show()

## Merging

- **Left:** merging off, every selected loop is its own region.
- **Middle:** merging with the waist test (the default).
- **Right:** for comparison only, the waist test switched off (`notch_depth=inf`): every pair of pieces sharing a
  thin wall is merged.

Merged regions are orange. `postprocess` repeats only the merging, without rebuilding the forests.

In [ ]:
variants = {
    "merging off": nucseg.postprocess(segmentation, merge=False),
    "merging (default)": segmentation,
    "merging, no waist test": nucseg.postprocess(segmentation, notch_depth=math.inf),
}
fig, axes = plt.subplots(1, 3, figsize=(21, 7))
for ax, (label, version) in zip(axes, variants.items()):
    plots.plot_regions(ax, version, title=f"{label} ({len(version.nuclei)} nuclei)")
fig.tight_layout()
plt.show()

## Nuclei

Labels match the overlays. `z_window`: slices of the projection the outline comes from; `pieces`: loops merged into
the nucleus; `notches`: for a merged nucleus, the most pinches at a wall between its pieces; `width_um`: twice the
radius of the largest disk inside.

In [ ]:
pd.DataFrame(segmentation.table()).set_index("label")

## Outline cleaning and nuclei at the image edge (optional)

Left: the default. Middle: `clean_outlines=True`; areas change by a few percent and spurs into the ER are cut.
Right: `border_frame=True`; nuclei cut by the image edge (magenta) are found too. Their area is only the visible
part, so leave them out of size statistics (`border` column).

In [ ]:
cleaned = nucseg.segment_2d(stack, replace(parameters, clean_outlines=True))
framed = nucseg.segment_2d(stack, replace(parameters, border_frame=True))
fig, axes = plt.subplots(1, 3, figsize=(21, 7))
plots.plot_regions(axes[0], segmentation, title=f"default ({len(segmentation.nuclei)} nuclei)")
plots.plot_regions(axes[1], cleaned, title="clean_outlines=True")
n_border = sum(r.border for r in framed.nuclei)
plots.plot_regions(axes[2], framed, title=f"border_frame=True ({len(framed.nuclei) - n_border} + {n_border} at the edge)")
fig.tight_layout()
plt.show()

## Tuning

| symptom | parameter |
|---|---|
| a nucleus is missing, its ring is faint or broken | lower `min_lifetime`; raise `smooth_um` a little |
| dark pockets in the ER are taken for nuclei | raise `min_lifetime`, `min_solidity` or `min_width_um` |
| small or large nuclei are dropped | `min_area_um2`, `max_area_um2`, `min_width_um` |
| one nucleus is split in two | raise `wall_um`; check `pieces`/`notches` |
| two touching nuclei are merged | lower `wall_um` or `notch_depth`, or `merge=False` |
| outlines have spurs into the ER | `clean_outlines=True` |
| outlines too far out / in | lower / raise `cycle_position` |
| slow | raise `bin_factor`; narrow `z_first`..`z_last` |